# Voice Cloning (Part 1)

Voice cloning experiments and their Word Error Rate, using TIMIT (Kaggle copy) and the pretrained SV2TTS models in `WebApp/saved_models/default`.

Notes:
- TIMIT audio is read from the converted `*.WAV.wav` files (the `.WAV` files are NIST SPHERE)
- Sentence ids such as `(sa1)` are stripped from the TIMIT prompts
- Only `N_SPEAKERS` speakers are cloned in the final step (each clone takes about a minute on CPU). `clone_speakers.py` clones the 300 speakers used for fake audio detection
- Cloned files go to `data/cloned_audio`

The WER cells send audio to Google's free speech recognition API, so they need internet access.

# Voice Cloning and Fake Audio Detection

### Introduction
A technology company working in the Cyber Security industry wants to build data driven systems that help individuals and organizations to understand whether audio and video media is authentic or fake.


### Data Description:

The following dataset is used in this project:

[**TIMIT Dataset:**](https://github.com/philipperemy/timit) The TIMIT corpus of read speech is designed to provide speech data for acoustic-phonetic studies and for the development and evaluation of automatic speech recognition systems. TIMIT contains a total of 6300 sentences, 10 sentences spoken by each of 630 speakers from 8 major dialect regions of the United States.

### Goals(s):
The goal of this project is to build algorithms that can synthesize spoken audio by converting a speaker’s voice to another speaker’s voice with the end goal to detect if any spoken audio is pristine or fake.

### Success Metrics:
* **Voice cloning (VC):** Word Error Rate (WER)
* **Fake audio detection (FAD):** F-score

### Methodology:
**Step 1:** We will build a **Voice Cloning system (VC)** that clones source speaker's voice to the target speaker's voice. We will utlize the TIMIT dataset for this step as it consists of aligned text-audio data with various speakers.

**Step 2:** We will build a **Fake audio detection system (FAD)** that detects if a given audio is synthetically generated or not. Real recordings are used as positive examples, and negative examples are created using the voice cloning system as an automatic data/label generator.

## Part 1: Voice Cloning

As mentioned above, the TIMIT dataset contains 10 recordings from 630 different speakers. All the audio files (4s) are recordings of short sentences from different users. For our Voice cloning system, we will use SV2TTS deep learning framework which is effective in converting text to speech. The framework is made of 3 stages. In the first stage, one creates a digital representation of a voice from a few seconds of audio. In the second and third stages, this representation is used as reference to generate speech given arbitrary text. We will evaluate the performance of the voice cloning system by performing the following experiments. 

We will use the following github repo (https://github.com/CorentinJ/Real-Time-Voice-Cloning.git) for the voice cloning system

**Exp 1:** We will utilize the short recordings from the TIMIT dataset to generate short and long audios and evaluate the perfromance of the model.

**Exp 2:** We will combine the recordings from similar users and use that to generate short and synthetic audios and evaluate the model's perfromance.


### Initial Setup
Import relevant libraries and install the voice cloning model from the github repo

In [1]:
import os
import re
from pathlib import Path
import librosa
import librosa.display
import pandas as pd
import functions
import numpy as np
from IPython.display import display, Audio, clear_output
from moviepy.editor import concatenate_audioclips, AudioFileClip
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

N_SPEAKERS = 10  # number of speakers cloned in the final step

for d in ['cloned_data', 'data/long_version_audio', 'data/cloned_audio']:
    os.makedirs(d, exist_ok=True)

ModuleNotFoundError: No module named 'librosa'

The following cells contain information about the recordings from the TIMIT dataset

In [ ]:
train_csv = pd.read_csv("TIMIT_orig/DOC/train_data.csv")
print(train_csv.shape)
train_csv.head()

In [ ]:
test_csv = pd.read_csv("TIMIT_orig/DOC/test_data.csv")
print(test_csv.shape)
test_csv.head()

The following texts will be used to genrate the synthetic audio files

In [ ]:
long_text = """Two roads diverged in a yellow wood,
And sorry I could not travel both 
And be one traveler, long I stood
And looked down one as far as I could
To where it bent in the undergrowth;""".replace('\n',' ')

short_text = "If you're looking for random paragraphs, you've come to the right place"

We will display random samples from the TIMIT dataset

In [ ]:
audio_1, sample_rate = librosa.load('TIMIT_orig/TRAIN/DR4/FLHD0/SA1.WAV.wav')
display(Audio(audio_1, autoplay=True, rate=sample_rate))

In [ ]:
audio_2, sample_rate = librosa.load('TIMIT_orig/TRAIN/DR1/MCPM0/SA2.WAV.wav')
display(Audio(audio_2, autoplay=True, rate=sample_rate))

In [ ]:
audio_3, sample_rate = librosa.load('TIMIT_orig/TRAIN/DR5/MDSJ0/SA1.WAV.wav')
display(Audio(audio_3, rate=sample_rate))

#### Exp - 1.1: 
In the following section, we will use the short recordings from the TIMIT dataset to generate the synthetic audios for short texts.

In [ ]:
functions.synthesize_audio(audio_1, short_text, 'cloned_data/exp-1-1.wav')
functions.synthesize_audio(audio_2, short_text, 'cloned_data/exp-1-2.wav')
functions.synthesize_audio(audio_3, short_text, 'cloned_data/exp-1-3.wav')

In [ ]:
display(Audio('cloned_data/exp-1-1.wav', autoplay=True))

In [ ]:
display(Audio('cloned_data/exp-1-2.wav', autoplay=True))

In [ ]:
sim_score11, wer11 = functions.text_comparison('cloned_data/exp-1-1.wav', short_text)
sim_score12, wer12 = functions.text_comparison('cloned_data/exp-1-2.wav', short_text)
sim_score13, wer13 = functions.text_comparison('cloned_data/exp-1-3.wav', short_text)

print("WER for the cloned audio: ", (wer11, wer12, wer13))
print("Mean WER: ", np.mean([wer11, wer12, wer13])*100)
print("Similarity scores for the cloned audio: ", (sim_score11, sim_score12, sim_score13))

#### Exp - 1.2: 
In the following section, we will use the short recordings from the TIMIT dataset to generate the synthetic audios for long texts.

In [ ]:
functions.synthesize_audio(audio_1, long_text, 'cloned_data/exp-2-1.wav')
functions.synthesize_audio(audio_2, long_text, 'cloned_data/exp-2-2.wav')
functions.synthesize_audio(audio_3, long_text, 'cloned_data/exp-2-3.wav')

In [ ]:
display(Audio('cloned_data/exp-2-1.wav', autoplay=True))

In [ ]:
display(Audio('cloned_data/exp-2-3.wav', autoplay=True))

In [ ]:
sim_score21, wer21 = functions.text_comparison('cloned_data/exp-2-1.wav', long_text)
sim_score22, wer22 = functions.text_comparison('cloned_data/exp-2-2.wav', long_text)
sim_score23, wer23 = functions.text_comparison('cloned_data/exp-2-3.wav', long_text)

print("WER for the cloned audio: ", (wer21, wer22, wer23))
print("Mean WER: ", np.mean([wer21, wer22, wer23])*100)
print("Similarity scores for the cloned audio: ", (sim_score21, sim_score22, sim_score23))

Results from exp 1-2 are much worse than the results from exp 1-1. This is because, it is difficult to generate long audio files from short sample and the rest of the words could not be synthesised

For the next part of the experiments, we will combine recoedings from the same speakers in the TIMIT dataset into a single file (30s). There are 6300 recordings from 630 speakers. We will generate 630 audio files by combining these recordings from the same speakers.

In [ ]:
# This dictionary saves paths of all the recordings from the same speaker under a single key
my_dict = {}
for split in ['TIMIT_orig/TRAIN', 'TIMIT_orig/TEST']:
    for file_path in sorted(Path(split).glob('*/*/*.WAV.wav')):
        my_dict.setdefault(file_path.parent.name, []).append(str(file_path))

print(len(my_dict))
print(list(my_dict.items())[0])

Concatenate the audio files from the same speakers into a single file 

In [ ]:
functions.concat_audio(my_dict['MWDK0'], 'cloned_data/long-audio-1.wav')
functions.concat_audio(my_dict['FGRW0'], 'cloned_data/long-audio-2.wav')
functions.concat_audio(my_dict['MSAT1'], 'cloned_data/long-audio-3.wav')

In [ ]:
display(Audio('cloned_data/long-audio-1.wav', autoplay=True))

#### Exp - 2.1: 
In the following section, we will use the long recordings from the TIMIT dataset to generate the synthetic audios for short texts.

In [ ]:
functions.synthesize_audio('cloned_data/long-audio-1.wav', short_text, 'cloned_data/exp-3-1.wav')
functions.synthesize_audio('cloned_data/long-audio-2.wav', short_text, 'cloned_data/exp-3-2.wav')
functions.synthesize_audio('cloned_data/long-audio-3.wav', short_text, 'cloned_data/exp-3-3.wav')

In [ ]:
display(Audio('cloned_data/exp-3-1.wav', autoplay=True))

In [ ]:
display(Audio('cloned_data/exp-3-2.wav', autoplay=True))

In [ ]:
sim_score31, wer31 = functions.text_comparison('cloned_data/exp-3-1.wav', short_text)
sim_score32, wer32 = functions.text_comparison('cloned_data/exp-3-2.wav', short_text)
sim_score33, wer33 = functions.text_comparison('cloned_data/exp-3-3.wav', short_text)

print("WER for the cloned audio: ", (wer31, wer32, wer33))
print("Mean WER: ", np.mean([wer31, wer32, wer33])*100)
print("Similarity scores for the cloned audio: ", (sim_score31, sim_score32, sim_score33))

#### Exp - 2.2: 
In the following section, we will use the long recordings from the TIMIT dataset to generate the synthetic audios for long texts.

In [ ]:
functions.synthesize_audio('cloned_data/long-audio-1.wav', long_text, 'cloned_data/exp-4-1.wav')
functions.synthesize_audio('cloned_data/long-audio-2.wav', long_text, 'cloned_data/exp-4-2.wav')
functions.synthesize_audio('cloned_data/long-audio-3.wav', long_text, 'cloned_data/exp-4-3.wav')

In [ ]:
display(Audio('cloned_data/exp-4-1.wav', autoplay=True))

In [ ]:
display(Audio('cloned_data/exp-4-2.wav', autoplay=True))

In [ ]:
sim_score41, wer41 = functions.text_comparison('cloned_data/exp-4-1.wav', long_text)
sim_score42, wer42 = functions.text_comparison('cloned_data/exp-4-2.wav', long_text)
sim_score43, wer43 = functions.text_comparison('cloned_data/exp-4-3.wav', long_text)

print("WER for the cloned audio: ", (wer41, wer42, wer43))
print("Mean WER: ", np.mean([wer41, wer42, wer43])*100)
print("Similarity scores for the cloned audio: ", (sim_score41, sim_score42, sim_score43))

The results from exp 2-2 and exp 1-2 indicate the model is not efficient in generating long audio files. It is better to generate sythetic audio files for short texts.

For our project, we will generate short synthetic audio files and use them in our classification model.

In the first step, we will combine all the recordings from the same speakers into a single file. Only the first `N_SPEAKERS` speakers are used here.

In [ ]:
# Combine all the recordings from same speakers into a single file

speakers = sorted(my_dict)[:N_SPEAKERS]

for key in speakers:
    files = my_dict[key]
    clips = [AudioFileClip(clip) for clip in files]
    final_clip = concatenate_audioclips(clips)
    final_clip.write_audiofile(f"data/long_version_audio/{key}.wav", logger=None)

text_list is a list of samples short texts used in the TIMIT dataset

In [ ]:
text_list = []

with open('TIMIT_orig/DOC/PROMPTS.TXT', 'r') as text_file:
    for line in text_file:
        if line.startswith(';'):
            continue
        a = re.sub(r'\s*\(\w+\)\s*$', '', line.strip())  # drop the sentence id, e.g. (sa1)
        a = a.replace('.', '').replace('?', '').replace('!', '')
        if len(a.split()) >= 11:
            text_list.append(a)

print(len(text_list), text_list[:3])

We will generate synthetic audio files

In [ ]:
file_count = 1

for index, file in enumerate(sorted(os.listdir("data/long_version_audio"))):
    text = text_list[index]
    src = os.path.join("data/long_version_audio", file)
    dst = os.path.join("data/cloned_audio", file)
    functions.synthesize_audio(src, text, dst)
    print(f"{file_count} files have been cloned.")
    file_count+=1

Sample synthetic file

In [ ]:
sample_file = os.path.join('data/cloned_audio', sorted(os.listdir('data/cloned_audio'))[0])
display(Audio(sample_file))

In [ ]:
data_example, sampling_rate = librosa.load(sample_file)

fig1 = plt.figure(figsize=(12, 3))
librosa.display.waveshow(data_example, sr=sampling_rate)
plt.title('Original wav file')

fig2 = plt.figure(figsize=(12, 3))
mfcc = librosa.feature.mfcc(y=data_example, sr=sampling_rate, n_mfcc=40)
librosa.display.specshow(mfcc, sr=sampling_rate, x_axis='time')
plt.colorbar()
plt.title('Mel-frequency cepstral coefficients (MFCCs)');